# A7 Part 2 – Retrieval-Augmented Generation with our Small Language Model
**Team 4 – GreenHand.** This notebook applies RAG to **the same SLM** trained in Part 1 (`checkpoints/slm_baseline.pt`): the class Transformer
(embedding 48, 2 heads, 1 layer, feed-forward 192, context **32 characters**), trained for 30,000 steps on `greenhand_simple_corpus.txt`.
No pretrained Hugging Face model is used, and **RAG never changes the model's weights** (verified at the end with a SHA-256 hash).

**What is reused from class:** the retrieval pipeline of the Week 6 RAG notebook (TF-IDF vectors + cosine similarity, `retrieve()`, a prompt
builder, a baseline-vs-RAG comparison) and the SLM code of Week 7 (`TransformerBlock`, `TinyTransformerLM`, `generate_text`, copied unchanged).

**Compatibility adjustments for the SLM** (the only changes):
* Ollama is replaced by `ask_slm()`, which calls the SLM's own `generate_text`.
* The SLM is *character-level with a 30-character vocabulary* (`a–z` except `j`, space, newline, `,` `.` `'`), so every document, question and prompt
  is lower-case, without digits, `?` or `:` – numbers are written as words. All texts are validated against the checkpoint's vocabulary.
* The SLM can only see its last **32 characters**, so the class's long instruction prompt is replaced by a minimal one (`question … answer`), and
  we record exactly which part of each prompt the model could see.
* The class retrieval threshold (`min_score`) is set to 0 so that Top-K really returns K documents.

In [1]:
import hashlib, json, math, random, re, time, warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 90); pd.set_option("display.max_rows", 200)

warnings.filterwarnings("ignore", category=UserWarning)     # silence PyTorch's internal attention-mask notice
SEED = 42
DEVICE = torch.device("cpu")                      # same as the class notebook; the tiny model needs no GPU
torch.set_num_threads(2)
CHECKPOINT_PATH = Path("checkpoints/slm_baseline.pt")   # the SLM selected in Part 1 (git-ignored, never committed)
CORPUS_PATH = Path("greenhand_simple_corpus.txt")
RESULTS = Path("results"); RESULTS.mkdir(exist_ok=True)
print("PyTorch", torch.__version__, "| device", DEVICE)

PyTorch 2.0.1 | device cpu


## Load the trained SLM from Part 1
The two classes and `generate_text` below are **copied unchanged** from the class SLM notebook (Cells 9A, 9B and 12A).

In [2]:
class TransformerBlock(nn.Module):
    def __init__(self, embedding_dim, num_heads, feedforward_dim):
        super().__init__()
        self.attention = nn.MultiheadAttention(embed_dim=embedding_dim, num_heads=num_heads, dropout=0.0, batch_first=True)
        self.ln1 = nn.LayerNorm(embedding_dim)
        self.ln2 = nn.LayerNorm(embedding_dim)
        self.feedforward = nn.Sequential(nn.Linear(embedding_dim, feedforward_dim), nn.ReLU(), nn.Linear(feedforward_dim, embedding_dim))

    def forward(self, x):
        T = x.shape[1]
        mask = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1)
        normalized = self.ln1(x)
        attended, _ = self.attention(normalized, normalized, normalized, attn_mask=mask, need_weights=False)
        x = x + attended
        x = x + self.feedforward(self.ln2(x))
        return x


class TinyTransformerLM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, num_heads, num_layers, feedforward_dim, context_length):
        super().__init__()
        self.context_length = context_length
        self.token_embedding = nn.Embedding(vocab_size, embedding_dim)
        self.position_embedding = nn.Embedding(context_length, embedding_dim)
        self.blocks = nn.Sequential(*[TransformerBlock(embedding_dim, num_heads, feedforward_dim) for _ in range(num_layers)])
        self.ln_final = nn.LayerNorm(embedding_dim)
        self.output = nn.Linear(embedding_dim, vocab_size)

    def forward(self, input_ids):
        B, T = input_ids.shape
        if T > self.context_length:
            raise ValueError("Input sequence exceeds context length.")
        positions = torch.arange(T, device=input_ids.device)
        x = self.token_embedding(input_ids) + self.position_embedding(positions)
        x = self.blocks(x)
        x = self.ln_final(x)
        return self.output(x)


@torch.no_grad()
def generate_text(model, prompt, max_new_tokens=100, temperature=1.0):
    if temperature <= 0:
        raise ValueError("Temperature must be greater than zero.")
    was_training = model.training
    model.eval()
    ids = torch.tensor([encode(prompt)], dtype=torch.long, device=DEVICE)
    for _ in range(max_new_tokens):
        context = ids[:, -model.context_length:]           # <-- only the last context_length characters are visible
        logits = model(context)
        next_logits = logits[:, -1, :] / temperature
        probabilities = F.softmax(next_logits, dim=-1)
        next_id = torch.multinomial(probabilities, num_samples=1)
        ids = torch.cat([ids, next_id], dim=1)
    if was_training:
        model.train()
    return decode(ids[0].tolist())

In [3]:
saved = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=True)
cfg = saved["config"]
chars = saved["vocab"]                                     # the vocabulary comes from the checkpoint, not from our new documents
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
encode = lambda s: [stoi[ch] for ch in s]
decode = lambda ids: "".join(itos[int(i)] for i in ids)

slm = TinyTransformerLM(vocab_size=len(chars), **cfg).to(DEVICE)
slm.load_state_dict(saved["model_state"])
slm.eval()
for p in slm.parameters():
    p.requires_grad_(False)                                # RAG must not train the model

CONTEXT = cfg["context_length"]


def weights_hash(model):
    h = hashlib.sha256()
    for name, tensor in sorted(model.state_dict().items()):
        h.update(name.encode()); h.update(tensor.detach().cpu().numpy().tobytes())
    return h.hexdigest()


HASH_BEFORE = weights_hash(slm)
print("architecture:", cfg)
print("trainable parameters:", f"{sum(p.numel() for p in slm.parameters()):,}")
print("vocabulary:", len(chars), "characters ->", repr("".join(chars)))
print("weights SHA-256 (before RAG):", HASH_BEFORE[:16], "...")

architecture: {'embedding_dim': 48, 'num_heads': 2, 'num_layers': 1, 'feedforward_dim': 192, 'context_length': 32}
trainable parameters: 32,814
vocabulary: 30 characters -> "\n ',.abcdefghiklmnopqrstuvwxyz"
weights SHA-256 (before RAG): 27cd5626bc3dc6e5 ...


## Step 2.1 – A new knowledge base (5 pts)
A fictional garden-supplies web shop, **"sprout shop"**. The original SLM corpus is about growers, plants, gardens and the GreenHand app; it never mentions
deliveries, prices, returns, members, refunds or support. Seven short documents, each with a unique id. They contain no private information.
They are written in the SLM's alphabet (numbers as words) and are kept short because the model sees only 32 characters.

In [4]:
KNOWLEDGE_BASE = [
    {"id": "D1", "title": "delivery fee",      "text": "delivery costs two dollars."},
    {"id": "D2", "title": "free delivery",     "text": "orders over forty dollars get free delivery."},
    {"id": "D3", "title": "return policy",     "text": "you can return items within seven days."},
    {"id": "D4", "title": "member benefits",   "text": "members get ten percent off every order."},
    {"id": "D5", "title": "support hours",     "text": "support is open from nine to five."},
    {"id": "D6", "title": "shop closing day",  "text": "sprout shop closes on sundays."},
    {"id": "D7", "title": "refund timing",     "text": "refunds arrive within three days."},
]
assert len({d["id"] for d in KNOWLEDGE_BASE}) == len(KNOWLEDGE_BASE) >= 5

corpus_text = CORPUS_PATH.read_text(encoding="utf-8")
corpus_words = set(re.findall(r"[a-z']+", corpus_text))
corpus_lines = set(corpus_text.split("\n"))


def check_in_vocab(text, what):
    bad = sorted(set(text) - set(chars))
    assert not bad, f"{what} contains characters the SLM cannot represent: {bad}"


kb_rows = []
for d in KNOWLEDGE_BASE:
    check_in_vocab(d["text"], d["id"]); check_in_vocab(d["title"], d["id"])
    new_words = [w for w in re.findall(r"[a-z']+", d["text"]) if w not in corpus_words]
    kb_rows.append({"id": d["id"], "title": d["title"], "text": d["text"], "chars": len(d["text"]),
                    "sentence in training corpus?": d["text"] in corpus_lines,
                    "words never seen in training corpus": ", ".join(new_words)})
display(pd.DataFrame(kb_rows))
print("All documents use only the SLM vocabulary; none of the sentences occurs in the training corpus.")
_ = Path("results/knowledge_base.json").write_text(json.dumps(KNOWLEDGE_BASE, indent=1), encoding="utf-8")

,id,title,text,chars,sentence in training corpus?,words never seen in training corpus
0,D1,delivery fee,delivery costs two dollars.,27,False,"delivery, costs, dollars"
1,D2,free delivery,orders over forty dollars get free delivery.,44,False,"orders, over, dollars, get, free, delivery"
2,D3,return policy,you can return items within seven days.,39,False,"you, return, items, within, seven"
3,D4,member benefits,members get ten percent off every order.,40,False,"members, get, ten, percent, order"
4,D5,support hours,support is open from nine to five.,34,False,"support, open, nine"
5,D6,shop closing day,sprout shop closes on sundays.,30,False,"shop, closes, sundays"
6,D7,refund timing,refunds arrive within three days.,33,False,"refunds, arrive, within"


All documents use only the SLM vocabulary; none of the sentences occurs in the training corpus.


## Step 2.2 – The SLM without RAG (5 pts)
Five questions about the knowledge base. **Generation settings are fixed for every experiment in this assignment** (they are reused unchanged in Parts 3 and 4):
temperature 0.2 (near-greedy, so differences between conditions come from the prompt, not from sampling noise), 60 new characters, `torch.manual_seed(42)` before every call.
The *response* is the first non-empty line the model writes after the prompt.

In [5]:
TEMPERATURE, MAX_NEW_TOKENS = 0.2, 60

QUESTIONS = [   # gold = the document that contains the answer; fact = the words that carry the answer; topic = words a relevant reply would use
    {"q": "Q1", "text": "how much does delivery cost", "gold": "D1", "fact": ["two"],            "topic": ["delivery", "cost", "costs", "dollars"]},
    {"q": "Q2", "text": "when is delivery free",       "gold": "D2", "fact": ["forty"],          "topic": ["delivery", "free", "orders", "over", "dollars"]},
    {"q": "Q3", "text": "how long can i return an item", "gold": "D3", "fact": ["seven"],        "topic": ["return", "items", "item", "days", "within"]},
    {"q": "Q4", "text": "what do members get",         "gold": "D4", "fact": ["ten", "percent"], "topic": ["members", "percent", "off", "order"]},
    {"q": "Q5", "text": "when is support open",        "gold": "D5", "fact": ["nine", "five"],   "topic": ["support", "open", "from", "hours"]},
]
for item in QUESTIONS:
    check_in_vocab(item["text"], item["q"])


def baseline_prompt(question):
    return f"question {question}.\nanswer "


def ask_slm(prompt, seed=SEED):
    """Reset the random seed, let the SLM continue the prompt, return (response, raw continuation)."""
    torch.manual_seed(seed)
    full = generate_text(slm, prompt, max_new_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE)
    continuation = full[len(prompt):]
    lines = [l.strip() for l in continuation.split("\n") if l.strip()]
    return (lines[0] if lines else "")[:80], continuation


baseline_rows = []
for item in QUESTIONS:
    prompt = baseline_prompt(item["text"])
    response, _ = ask_slm(prompt)
    baseline_rows.append({"Question": item["q"], "Question text": item["text"], "SLM response without RAG": response})
baseline_df = pd.DataFrame(baseline_rows)
display(baseline_df)
_ = Path("results/eval_questions.json").write_text(json.dumps(QUESTIONS, indent=1), encoding="utf-8")
_ = Path("results/part2_baseline_responses.json").write_text(json.dumps(
    {"settings": {"seed": SEED, "temperature": TEMPERATURE, "max_new_tokens": MAX_NEW_TOKENS, "prompt_template": "question {q}.\\nanswer "},
     "responses": baseline_rows}, indent=1), encoding="utf-8")

,Question,Question text,SLM response without RAG
0,Q1,how much does delivery cost,felor tracks scallions in small pots.
1,Q2,when is delivery free,felor tracks scallions in small pots.
2,Q3,how long can i return an item,from plant chili plants at home.
3,Q4,what do members get,from plant chili plants at home.
4,Q5,when is support open,felor tracks scallions in small pots.


## Step 2.3 – Apply RAG to the same SLM (10 pts)
**Retrieval = the class pipeline:** TF-IDF vectors (`TfidfVectorizer(stop_words="english", ngram_range=(1, 2))`) of *title + text* and cosine similarity to the question.
**Augmentation:** the retrieved documents are put in front of the question. The best document is placed *last* (right before the question) so that it has the best chance to
stay inside the model's 32-character window. **Generation:** the same SLM, same seed and settings as the baseline.

In [6]:
vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
document_matrix = vectorizer.fit_transform([d["title"] + " " + d["text"] for d in KNOWLEDGE_BASE])
print("document matrix:", document_matrix.shape, "(documents x vocabulary features)")


def retrieve(query, top_k):
    """Class retrieve(), with min_score = 0 so that Top-K always returns K documents."""
    scores = cosine_similarity(vectorizer.transform([query]), document_matrix).ravel()
    order = scores.argsort()[::-1][:top_k]
    return [dict(KNOWLEDGE_BASE[i], score=float(scores[i])) for i in order]


def build_rag_prompt(question, matches, order="docs_first"):
    docs = "\n".join(d["text"] for d in reversed(matches))      # reversed -> best document closest to the question
    if order == "docs_first":
        prompt = f"{docs}\nquestion {question}.\nanswer "
    else:                                                       # ablation: question first, documents just before the answer
        prompt = f"question {question}.\n{docs}\nanswer "
    check_in_vocab(prompt, "prompt")
    return prompt


# retrieval quality on its own, independent of the SLM
ret = []
for item in QUESTIONS:
    row = {"Question": item["q"], "gold document": item["gold"]}
    for k in (1, 2, 3, 5):
        ids = [d["id"] for d in retrieve(item["text"], k)]
        row[f"Top-{k}"] = ", ".join(ids) + ("  ✔" if item["gold"] in ids else "  ✘")
    ret.append(row)
display(pd.DataFrame(ret))
recall = {k: np.mean([item["gold"] in [d["id"] for d in retrieve(item["text"], k)] for item in QUESTIONS]) for k in (1, 2, 3, 5)}
print("recall@k of the retriever:", recall)

document matrix: (7, 59) (documents x vocabulary features)


,Question,gold document,Top-1,Top-2,Top-3,Top-5
0,Q1,D1,D1 ✔,"D1, D2 ✔","D1, D2, D7 ✔","D1, D2, D7, D6, D5 ✔"
1,Q2,D2,D2 ✔,"D2, D1 ✔","D2, D1, D7 ✔","D2, D1, D7, D6, D5 ✔"
2,Q3,D3,D3 ✔,"D3, D7 ✔","D3, D7, D6 ✔","D3, D7, D6, D5, D4 ✔"
3,Q4,D4,D4 ✔,"D4, D7 ✔","D4, D7, D6 ✔","D4, D7, D6, D5, D3 ✔"
4,Q5,D5,D5 ✔,"D5, D7 ✔","D5, D7, D6 ✔","D5, D7, D6, D4, D3 ✔"


recall@k of the retriever: {1: 1.0, 2: 1.0, 3: 1.0, 5: 1.0}


### The scoring rubric (response quality 1–10)
Quality is scored by a fixed rubric so that all 40 responses are judged the same way (there is a blank `manual_override` column in the CSV for any score the team wants to change after reading the text):
* **factual accuracy (weight 0.5):** share of the answer words (e.g. *two*, *forty*, *seven*) that appear in the response;
* **relevance (0.25):** whether the response talks about the question's topic (at least two topic words = full credit);
* **support from retrieved information (0.25):** share of the response's content words that occur in the retrieved documents (0 for the no-RAG baseline).
`score = round(1 + 9 × weighted sum)`, so an answer that is wrong, off-topic and unsupported gets 1.

In [7]:
STOP = set("the a an is are to of in on for and can you get do does how what when much long i it from every at with who will be".split())


def content_words(text):
    return [w for w in re.findall(r"[a-z']+", text) if w not in STOP]


def score_response(item, response, matches):
    words = set(re.findall(r"[a-z']+", response))
    accuracy = sum(w in words for w in item["fact"]) / len(item["fact"])
    relevance = min(1.0, sum(w in words for w in item["topic"]) / 2)
    support = 0.0
    if matches:
        doc_words = set(w for d in matches for w in re.findall(r"[a-z']+", d["text"]))
        cw = content_words(response)
        support = (sum(w in doc_words for w in cw) / len(cw)) if cw else 0.0
    score = int(min(10, max(1, math.floor(1 + 9 * (0.5 * accuracy + 0.25 * relevance + 0.25 * support) + 0.5))))
    return accuracy, relevance, support, score


def visible(prompt):
    return prompt[-CONTEXT:]


BASE_RESPONSE = {r["Question"]: r["SLM response without RAG"] for r in baseline_rows}
rows = []
for item in QUESTIONS:
    for k in (1, 2, 3, 5):
        matches = retrieve(item["text"], k)
        prompt = build_rag_prompt(item["text"], matches)
        response, _ = ask_slm(prompt)
        acc, rel, sup, score = score_response(item, response, matches)
        gold_text = next(d["text"] for d in KNOWLEDGE_BASE if d["id"] == item["gold"])
        rows.append({"Question": item["q"], "Top-K": k, "Retrieved documents": ", ".join(d["id"] for d in matches),
                     "Prompt length (chars)": len(prompt), "SLM sees only (last 32 chars)": visible(prompt),
                     "gold document fully visible?": gold_text in visible(prompt),
                     "Generated response": response, "same as no-RAG response?": response == BASE_RESPONSE[item["q"]], "accuracy": round(acc, 2), "relevance": round(rel, 2), "support": round(sup, 2),
                     "Response Quality (1-10)": score, "manual_override": ""})
rag = pd.DataFrame(rows)
rag.to_csv(RESULTS / "part2_rag_results.csv", index=False)
display(rag.drop(columns=["manual_override"]))

,Question,Top-K,Retrieved documents,Prompt length (chars),SLM sees only (last 32 chars),gold document fully visible?,Generated response,same as no-RAG response?,accuracy,relevance,support,Response Quality (1-10)
0,Q1,1,D1,73,much does delivery cost.\nanswer,False,felor tracks scallions in small pots.,True,0.0,0.0,0.0,1
1,Q1,2,"D1, D2",118,much does delivery cost.\nanswer,False,felor tracks scallions in small pots.,True,0.0,0.0,0.0,1
2,Q1,3,"D1, D2, D7",152,much does delivery cost.\nanswer,False,felor tracks scallions in small pots.,True,0.0,0.0,0.0,1
3,Q1,5,"D1, D2, D7, D6, D5",218,much does delivery cost.\nanswer,False,felor tracks scallions in small pots.,True,0.0,0.0,0.0,1
4,Q2,1,D2,84,n when is delivery free.\nanswer,False,felor tracks scallions in small pots.,True,0.0,0.0,0.0,1
5,Q2,2,"D2, D1",112,n when is delivery free.\nanswer,False,felor tracks scallions in small pots.,True,0.0,0.0,0.0,1
6,Q2,3,"D2, D1, D7",146,n when is delivery free.\nanswer,False,felor tracks scallions in small pots.,True,0.0,0.0,0.0,1
7,Q2,5,"D2, D1, D7, D6, D5",212,n when is delivery free.\nanswer,False,felor tracks scallions in small pots.,True,0.0,0.0,0.0,1
8,Q3,1,D3,87,ng can i return an item.\nanswer,False,from plant chili plants at home.,True,0.0,0.0,0.0,1
9,Q3,2,"D3, D7",121,ng can i return an item.\nanswer,False,from plant chili plants at home.,True,0.0,0.0,0.0,1


### Summary per retrieval configuration

In [8]:
base_scores = [score_response(item, r["SLM response without RAG"], [])[3] for item, r in zip(QUESTIONS, baseline_rows)]
summary = rag.groupby("Top-K").agg(mean_quality=("Response Quality (1-10)", "mean"), mean_accuracy=("accuracy", "mean"),
                                   mean_prompt_chars=("Prompt length (chars)", "mean"),
                                   gold_doc_fully_visible=("gold document fully visible?", "mean"),
                                   same_as_baseline=("same as no-RAG response?", "mean")).round(2)
summary.loc["no RAG"] = [np.mean(base_scores), 0.0, np.mean([len(baseline_prompt(i["text"])) for i in QUESTIONS]), np.nan, np.nan]
summary["recall@k"] = [recall.get(k, np.nan) for k in summary.index]
display(summary)

,mean_quality,mean_accuracy,mean_prompt_chars,gold_doc_fully_visible,same_as_baseline,recall@k
Top-K,,,,,,
1,1.0,0.0,79.0,0.0,1.0,1.0
2,1.0,0.0,114.0,0.0,1.0,1.0
3,1.0,0.0,146.2,0.0,1.0,1.0
5,1.0,0.0,219.0,0.0,1.0,1.0
no RAG,1.0,0.0,41.2,NaN,NaN,NaN


### How stable are these results? (many random seeds)
The table above uses one seed. To make sure the conclusions are not an accident of that seed, every condition is generated again with 20 different seeds.
The ablation row *question first* swaps the prompt order (question, then documents) to test the context-window explanation.

In [9]:
RESP_LOG = []


def hit_stats(order, k, n_seeds=20):
    hits_any = hits_all = quality = 0.0; n = 0
    for item in QUESTIONS:
        matches = retrieve(item["text"], k) if k else []
        prompt = baseline_prompt(item["text"]) if not k else build_rag_prompt(item["text"], matches, order)
        for s in range(n_seeds):
            resp, _ = ask_slm(prompt, seed=s)
            RESP_LOG.append(resp)
            words = set(re.findall(r"[a-z']+", resp))
            hits_any += any(w in words for w in item["fact"]); hits_all += all(w in words for w in item["fact"])
            quality += score_response(item, resp, matches)[3]; n += 1
    return {"answer word appears (any)": hits_any / n, "all answer words appear": hits_all / n, "mean quality (1-10)": quality / n}


stab = [{"condition": "no RAG", **hit_stats("docs_first", 0)}]
for order, label in (("docs_first", "RAG, documents first"), ("question_first", "RAG, question first")):
    for k in (1, 2, 3, 5):
        stab.append({"condition": f"{label}, Top-{k}", **hit_stats(order, k)})
stab = pd.DataFrame(stab).set_index("condition").round(3)
stab.to_csv(RESULTS / "part2_rag_robustness.csv")
display(stab)
common = Counter(RESP_LOG)
print(f"{len(RESP_LOG)} generations produced only {len(common)} different responses. Most common:")
for text, n in common.most_common(8):
    print(f"  {n:4d} x  {text!r}")
print("generations that contain any answer word (two / forty / seven / ten / percent / nine / five):",
      sum(any(w in set(re.findall(r"[a-z']+", r)) for w in ("two", "forty", "seven", "ten", "percent", "nine", "five")) for r in RESP_LOG))

,answer word appears (any),all answer words appear,mean quality (1-10)
condition,,,
no RAG,0.0,0.0,1.07
"RAG, documents first, Top-1",0.0,0.0,1.07
"RAG, documents first, Top-2",0.0,0.0,1.07
"RAG, documents first, Top-3",0.0,0.0,1.12
"RAG, documents first, Top-5",0.0,0.0,1.21
"RAG, question first, Top-1",0.0,0.0,1.13
"RAG, question first, Top-2",0.0,0.0,1.13
"RAG, question first, Top-3",0.0,0.0,1.25
"RAG, question first, Top-5",0.0,0.0,1.36


900 generations produced only 23 different responses. Most common:
   208 x  'from plant chili plants at home.'
   100 x  'from garden the balcony.'
    93 x  'few scallions in small pots.'
    59 x  'few scallions in a small yard.'
    48 x  'few sprout plants at home.'
    45 x  'spinach in vertical gardens.'
    44 x  'few sprout plants.'
    41 x  'from plant chili plants in vertical gardens.'
generations that contain any answer word (two / forty / seven / ten / percent / nine / five): 0


### Diagnostic: can this SLM copy text that is in its context at all?
RAG only works if the generator can *use* text placed in its prompt. We test the simplest form of that skill, **copying**, and keep everything inside the 32-character window:
`prompt = first 20 characters of a document + newline + its first 6 characters` (27 characters). A model that can use its context should continue with the remaining 14 characters.
As a control, the same cue follows the first 20 characters of a *different* document, where copying is impossible.

In [10]:
def copy_test(n_seeds=20):
    out = {"copy": [0, 0], "control": [0, 0]}      # [matching characters, characters]
    exact = {"copy": 0, "control": 0}; n = 0
    segs = [d["text"][:20] for d in KNOWLEDGE_BASE]
    for i, seg in enumerate(segs):
        cue, target = seg[:6], seg[6:]
        for name, context in (("copy", seg), ("control", segs[(i + 1) % len(segs)])):
            prompt = f"{context}\n{cue}"
            check_in_vocab(prompt, "copy test")
            for s in range(n_seeds):
                torch.manual_seed(s)
                cont = generate_text(slm, prompt, max_new_tokens=len(target), temperature=TEMPERATURE)[len(prompt):]
                out[name][0] += sum(a == b for a, b in zip(cont, target)); out[name][1] += len(target)
                exact[name] += cont == target
        n += n_seeds
    return pd.DataFrame({"characters copied correctly": {k: v[0] / v[1] for k, v in out.items()},
                         "whole continuation exact": {k: exact[k] / n for k in out}}).round(3)


copy_df = copy_test()
display(copy_df)
copy_df.to_csv(RESULTS / "part2_copy_diagnostic.csv")
example = KNOWLEDGE_BASE[0]["text"][:20]
torch.manual_seed(0)
print("example prompt:", repr(example + "\n" + example[:6]), "->", repr(generate_text(slm, example + "\n" + example[:6], max_new_tokens=14, temperature=TEMPERATURE)[27:]), "| expected:", repr(example[6:]))

,characters copied correctly,whole continuation exact
copy,0.090,0.0
control,0.085,0.0


example prompt: 'delivery costs two d\ndelive' -> ' sprouts.\nfami' | expected: 'ry costs two d'


### RAG did not change the model's weights

In [11]:
HASH_AFTER = weights_hash(slm)
reloaded = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=True)["model_state"]
identical = all(torch.equal(reloaded[k], v) for k, v in slm.state_dict().items())
print("SHA-256 before RAG:", HASH_BEFORE[:32])
print("SHA-256 after  RAG:", HASH_AFTER[:32])
print("weights identical to the saved Part 1 checkpoint:", identical, "| trainable parameters:", sum(p.requires_grad for p in slm.parameters()))
assert HASH_BEFORE == HASH_AFTER and identical

SHA-256 before RAG: 27cd5626bc3dc6e56921bc074c81114e
SHA-256 after  RAG: 27cd5626bc3dc6e56921bc074c81114e
weights identical to the saved Part 1 checkpoint: True | trainable parameters: 0


## Step 2.4 – Analysis (5 pts)
All numbers come from the tables above (5 questions × Top-1/2/3/5 = 20 RAG runs at seed 42, plus 900 generations over 20 seeds).

**1. Did providing retrieved information change the SLM's responses?**
No. At seed 42 every one of the 20 RAG responses is *identical* to the no-RAG response for the same question (`same as no-RAG response?` = 100% for every Top-K). Over 900 generations with 20 different seeds the SLM wrote only **23 different sentences**, all ordinary gardening text from its training corpus (for example `from plant chili plants at home.`), and none depended on which documents were retrieved. Three of the five questions even received the same sentence (`felor tracks scallions in small pots.`), so the reply is driven by the last few characters of the prompt (`…\nanswer `), not by the question or the documents.

**2. Did RAG improve factual accuracy?**
No. Accuracy is 0 in all 20 rows, and **0 of 900** seeded generations contain any of the answer words (*two, forty, seven, ten, percent, nine, five*). Mean quality is 1.0 with RAG and 1.0 without it at seed 42 (20-seed means: 1.07 without RAG, 1.07–1.21 with documents first, 1.13–1.36 with the question first). Those tiny differences are not correct answers: accuracy stays 0, and a score above 1 can only come from the relevance and support parts of the rubric (generic words that happen to occur in a document).
The retriever itself works perfectly (recall@1 = recall@5 = 100%), so the failure is in the generator, not in retrieval.

**3. Did retrieving more documents always improve the responses?**
No. Top-1 already retrieves the right document for all five questions, so Top-2, Top-3 and Top-5 only add distractor documents (D7, D6, D5 …) and lengthen the prompt from about 79 to 219 characters; responses and scores did not improve (quality 1.0 for every Top-K at seed 42; no answer word in any seed).

**4. How did the SLM's context-length limitation affect the results?**
The model sees only its last **32 characters**, but the prompts are 73–228 characters long, so 56–86% of every prompt is invisible to it. In the `SLM sees only` column the window contains just the end of the question and the cue (for example `much does delivery cost.\nanswer`); the document that holds the answer was **fully visible in 0 of 20 runs** – it is cut off before the model reads it, and with Top-3/5 the rest of the retrieved text is also cut off. Even the ablation that puts the question first and the documents just before `answer` (documents partly visible) produced no answer word in 400 generations, so the window is one cause but not the only one (see question 5).

**5. Why might RAG behave differently with a small character-level model than with a larger pretrained language model?**
RAG needs a generator that can *read* the supplied text and reuse it. Our SLM has 32,814 parameters, one Transformer layer, a 30-character vocabulary, and was trained only to continue 2,081 short gardening sentences. The copy diagnostic shows it cannot even repeat text that is inside its window: it reproduced **9.0%** of the characters after a copy cue versus **8.5%** in the control where copying is impossible (0 exact copies in 140 tries), and it writes unrelated corpus sentences instead. It also never saw words like *delivery* or *dollars* and has no notion of a question or an instruction. A large pretrained LLM has a context of thousands of tokens, subword tokens that cover new words, and billions of tokens of training that teach it to follow instructions and to answer from the text in its prompt, which is why RAG works there and not here.

**Limits of this experiment:** one trained checkpoint (original configuration, 30,000 steps) and one knowledge base; quality scores come from the rubric above, not from a human reading; the context length was not changed because all conditions must use the same checkpoint.